# **Import Library**

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import avg, col
from pyspark.sql.functions import month, year, sum as sum_
from pyspark.sql.functions import sum as sum_
from pyspark.sql.functions import col, sum as sum_, count
import matplotlib.pyplot as plt
from pyspark.sql.functions import sum
import pandas as pd
from pyspark.sql.functions import when
from pyspark.sql.functions import lit
import numpy as np
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.regression import LinearRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.evaluation import ClusteringEvaluator
import seaborn as sns
import mlflow

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:512)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:632)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$5(UsageLogging.scala:659)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:147)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:349)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:59)
	at com.databricks.logging.AttributionContext$.withValue(Att

# Data Loading and Import

In this step, the dataset is loaded into the Databricks environment using Apache Spark. The dataset is read from the specified file path in CSV format, with headers enabled and schema automatically inferred.

This process converts the raw dataset into a Spark DataFrame, which allows efficient distributed data processing and analysis.

In [0]:
df = spark.read.csv("/Workspace/Users/sushajayaweera79.com/Dataset/online_retail_II.csv", header=True, inferSchema=True)

# Dataset Overview

After loading the dataset, an initial exploration is performed to understand its structure, size, and attributes. This step is important to gain a clear understanding of the data before proceeding with preprocessing and analysis.

In [0]:
print("Row count:", df.count())
print("Column count:", len(df.columns))
print("Columns:", df.columns)

Row count: 1067371
Column count: 8
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [0]:
# Average sales per product
df_clean = df_clean.withColumnRenamed("Invoice", "InvoiceNo") \
                   .withColumnRenamed("Price", "UnitPrice") \
                   .withColumnRenamed("Customer ID", "CustomerID")

avg_sales_per_product = df_clean.groupBy("StockCode", "Description").agg(avg("Sales").alias("AvgSales")).orderBy("AvgSales", ascending=False)

In [0]:
display(avg_sales_per_product.limit(10))

StockCode,Description,AvgSales
23843,"PAPER CRAFT , LITTLE BIRDIE",168469.6
22502,PICNIC BASKET WICKER 60 PIECES,19809.75
22790,"MIRROR, ARCHED GEORGIAN",3884.0000000000005
15059A,ENGLISH ROSE EDWARDIAN PARASOL,1515.0
85220,SMALL FAIRY CAKE FRIDGE MAGNETS,1375.0
23131,MISELTOE HEART WREATH CREAM,996.0000000000001
84760L,LARGE HANGING GLASS+ZINC LANTERN,896.0
DOT,DOTCOM POSTAGE,744.1475
84964B,BLUE PAINTED KASHMIRI TABLE,495.0
22833,HALL CABINET WITH 3 DRAWERS,493.6323076923076


Databricks visualization. Run in Databricks to view.

## Most popular products

In [0]:
# Most popular products (by total quantity sold)
popular_products = df_clean.groupBy("StockCode", "Description").agg(sum_("Quantity").alias("TotalQuantity")).orderBy("TotalQuantity", ascending=False)

## Customer purchase patterns

In [0]:
# Customer purchase patterns (total sales and number of purchases per customer)
customer_patterns = df_clean.groupBy("CustomerID").agg({"Sales": "sum", "InvoiceNo": "count"}).withColumnRenamed("sum(Sales)", "TotalSales").withColumnRenamed("count(InvoiceNo)", "NumPurchases")

In [0]:
display(customer_patterns.limit(10 ))

CustomerID,TotalSales,NumPurchases
13457.0,148.22,9
13416.0,940.8,26
14286.0,8479.76,499
17047.0,2289.23,142
14285.0,3284.42,61
16782.0,10438.410000000007,1883
17866.0,1134.9499999999998,26
13050.0,12344.529999999999,1009
15184.0,689.95,85
17776.0,338.56,84


## Revenue trends over time

In [0]:
# Revenue trends over time (monthly revenue)
df_clean = df_clean.withColumn("Month", month(df_clean["InvoiceDate"])).withColumn("Year", year(df_clean["InvoiceDate"]))
monthly_revenue = df_clean.groupBy("Year", "Month").agg(sum_("Sales").alias("MonthlyRevenue")).orderBy("Year", "Month")

In [0]:
display(monthly_revenue.limit(10))

Year,Month,MonthlyRevenue
2009,12,686654.1599999949
2010,1,557319.0620000134
2010,2,506371.06600001536
2010,3,699608.9910000181
2010,4,594609.1919999976
2010,5,599985.7900000075
2010,6,639066.5800000058
2010,7,591636.7400000121
2010,8,604242.6499999989
2010,9,831615.0009999905


Databricks visualization. Run in Databricks to view.

## Classification Evaluation

In [0]:

# Classification Evaluation
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

results_classification = [
    ["DecisionTree", evaluator_acc.evaluate(dt_preds), evaluator_prec.evaluate(dt_preds), evaluator_rec.evaluate(dt_preds)],
    ["RandomForest", evaluator_acc.evaluate(rf_preds), evaluator_prec.evaluate(rf_preds), evaluator_rec.evaluate(rf_preds)],
    ["LogisticRegression", evaluator_acc.evaluate(lr_preds), evaluator_prec.evaluate(lr_preds), evaluator_rec.evaluate(lr_preds)]
]

In [0]:

# Regression Evaluation
reg_evaluator_mae = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="mae")
reg_evaluator_rmse = RegressionEvaluator(labelCol="Sales", predictionCol="prediction", metricName="rmse")
results_regression = [
    ["LinearRegression", reg_evaluator_mae.evaluate(lr_reg_preds), reg_evaluator_rmse.evaluate(lr_reg_preds)]
]

## Clustering Evaluation

In [0]:
# Clustering Evaluation
clust_evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction", metricName="silhouette")
wcss = kmeans_model.summary.trainingCost
silhouette = clust_evaluator.evaluate(kmeans_preds)
results_clustering = [
    ["KMeans", wcss, silhouette]
]

## Display evaluation tables

In [0]:
# Display evaluation tables
classification_df = pd.DataFrame(results_classification, columns=["Model", "Accuracy", "Precision", "Recall"])
regression_df = pd.DataFrame(results_regression, columns=["Model", "MAE", "RMSE"])
clustering_df = pd.DataFrame(results_clustering, columns=["Model", "WCSS", "SilhouetteScore"])

display(classification_df)
display(regression_df)
display(clustering_df)

Model,Accuracy,Precision,Recall
DecisionTree,0.9903856115704192,0.9904007071398473,0.9903856115704192
RandomForest,0.9891589482190589,0.9891673446649574,0.9891589482190589
LogisticRegression,0.9535608462319471,0.9491191134497607,0.9535608462319471


Model,MAE,RMSE
LinearRegression,13.506619500381658,217.01930050649318


Model,WCSS,SilhouetteScore
KMeans,1.066084019780523E10,0.9999950077298911


## Downlode the best accuracy model

In [0]:
# Log Spark ML model with MLflow using Unity Catalog volume
with mlflow.start_run():
    mlflow.spark.log_model(dt_model, "decision_tree_model", dfs_tmpdir="/Volumes/workspace/default/models")

2026/03/25 13:30:35 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.2) contains a local version label (+databricks.connect.18.0.2). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/25 13:30:38 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-42cf0026-8641-48e0-8d9b-4a/tmpwtnkgsw5/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/25 13:30:38 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l